In [ ]:
#@title Fault-Injector-14B · Stage 2 GPU 冒烟与 16K 显存标定（H100 / A100 兼容 · 40GB 预算红线）
import hashlib, os, re, subprocess, sys, tempfile
from pathlib import Path

for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN", "GH_TOKEN", "GITHUB_TOKEN", "WANDB_API_KEY"):
    os.environ.pop(key, None)
remote = "https://github.com/rrrrrredy/fault-injector.git"
release = "refs/tags/stage2b-gpu-v2"
tree_sha256 = "1a136f88d933dbcb18fb737804e0f1567cc78a33e116823d3ba05d91f3eea92b"
rev, tag = subprocess.check_output(["git", "ls-remote", "--refs", remote, release], text=True).strip().split()
if tag != release or not re.fullmatch(r"[0-9a-f]{40}", rev):
    raise RuntimeError("无法解析发布锚点的完整 Git SHA")
repo = Path(tempfile.mkdtemp(prefix="fi-code-", dir="/content"))
def run(*args):
    subprocess.run(args, cwd=repo, check=True)
run("git", "init", "--quiet")
run("git", "fetch", "--quiet", "--depth=1", remote, release)
run("git", "checkout", "--quiet", "--detach", "FETCH_HEAD")
pin = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
paths = ["fixtures", "materialize.py", "replay.py", "schemas", "stage0.lock.json", "stage1", "stage2"]
tree = subprocess.check_output(["git", "ls-tree", "-z", "HEAD", "--", *paths], cwd=repo)
if pin != rev or hashlib.sha256(tree).hexdigest() != tree_sha256:
    raise RuntimeError("Release commit or code tree digest mismatch")
print("Stage 2B Git SHA", pin, flush=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
from stage2.colab_assets import prepare
work = Path("/content/work_fi")
smoke_done = calibration_done = False
python = prepare(work)
run(python, "-m", "stage2.colab_grpo", "--mode", "mock-preflight")

In [ ]:
#@title 4B 冒烟 · 30 个优化器步与功能回归复核
smoke_done = calibration_done = False
run(python, "-m", "stage2.colab_grpo", "--mode", "smoke-4b")
smoke_done = True

In [ ]:
#@title 14B 标定 · 实际 16K 长度与 40,960 MiB 整卡峰值
calibration_done = False
run(python, "-m", "stage2.colab_grpo", "--mode", "calibrate-14b")
calibration_done = True

In [ ]:
#@title 归档 · 从 Google Drive 读回并验证两份快照
import json
from google.colab import drive
from stage2.colab_assets import collect_outputs, copy_and_verify

if not (smoke_done and calibration_done):
    raise RuntimeError("请先完成本次会话的两个 GPU 作业")
files = collect_outputs(work)
drive.mount("/content/drive")
destination = Path("/content/drive/MyDrive/fault_injector_2026/stage2_smoke") / (pin[:12] + "-" + files[0].parent.name)
restored = copy_and_verify(files, destination)
print(json.dumps({"directory": str(destination), "restored": restored}, indent=2))
print("4B Case 验收", json.loads((work / "smoke-4b_receipt.json").read_text())["case_status"])